In [2]:
import numpy as np
import pandas as pd
import random
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, fbeta_score, log_loss, confusion_matrix 
from sklearn.impute import KNNImputer
from imblearn.under_sampling import NearMiss 
from imblearn.over_sampling import BorderlineSMOTE
from skmultilearn.problem_transform import LabelPowerset
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import hamming_loss
from sklearn.metrics import classification_report as classification_report_multiout

import sys
import math
import shap
import xgboost as xgb

sys.path.insert(0, r"E:\Drive\NOA\MBD-Prediction\Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

import warnings
warnings.filterwarnings("ignore")

NUTS0 = 'GR'
P = 'P1'
p = P.lower()

file1 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\CMacedonia\GR_CMacedonia_WNV_Dataset_P1_2011_2022.csv'
file2 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\Attica\GR_Attica_WNV_Dataset_P1_2011_2022.csv'
file3 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\Thessaly\GR_Thessaly_WNV_Dataset_P1_2011_2022.csv'

dataset1 = read_data(file1)
dataset2 = read_data(file2)
dataset3 = read_data(file3)

dataset1.head()

dataset2.head()

dataset3.head()

dataset = pd.concat([dataset1, dataset2, dataset3])

dataset.reset_index(drop = True, inplace = True)

print(dataset.shape)

(204, 73)


In [3]:
def log_loss_vector(y_true, y_pred):
    e = 1e-15
    y_pred = np.clip(y_pred, e, 1 - e)
    log_loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return log_loss


def optimal_threashold_fbeta_vector(y_prob, y_true, beta = 1, clip_factor = 1e-15, round_factor = 2):
    if y_prob.shape != y_true.shape:
        raise Exception("Probability and Predictions matricies are not the same shape")

    threshold_per_month = []
    column_range = y_prob.shape[1]

    for col in range(column_range):
        precision, recall, thresholds = precision_recall_curve(y_true[:,col], y_prob[:,col])
        precision = np.clip(precision, clip_factor, 1 - clip_factor)
        recall = np.clip(recall, clip_factor, 1 - clip_factor)
        fb_score = ((1 + beta**2) * precision * recall) / ((beta**2 * precision) + recall)
        fb_max_index = np.argmax(fb_score)
        optimal_threshold_fb = round(thresholds[fb_max_index], ndigits = round_factor)
        threshold_per_month.append(optimal_threshold_fb)

    return np.array(threshold_per_month)

In [4]:
dataset.head()

,NUTS3_ID,year,x,y,ndvi_p1,ndwi_p1,ndmi_p1,ndbi_p1,lst_mean_p1,lst_day_mean_p1,lst_night_mean_p1,lst_min_p1,lst_max_p1,prec_mean_p1,prec_acc_p1,mosq_mean_p1,mosq_sum_p1,bio1,bio2,bio3,bio4,bio5,bio6,bio7,bio8,bio9,bio10,bio11,bio12,bio13,bio14,bio15,bio16,bio17,bio18,bio19,lc_prop1,lc_prop2,lc_prop3,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,distance_to_coast_std,distance_to_river_std,slope_mean_1km_std,aspect_mean_200m_std,elevation_mean_1km_std,hillshade_mean_1km_std,fs_area_1km_std,flow_accu_200m_std,males_lt15,males_15t64,males_gt65,males_total,females_lt15,females_15t64,females_gt65,females_total,mio_eur,l_total,unl_total,cases_total,cases_arr
0,EL521,2011,22.215876,40.552658,0.252698,0.164174,0.372403,-0.372403,6.208697,9.993104,2.424290,-5.821636,3.715091,3.286000,301.706854,433.757576,1301.272727,14.583333,18.666667,49.122807,768.952455,34.0,-4.0,38.0,9.166667,23.833333,23.833333,4.833333,1580.0,347.0,21.0,69.297060,585.0,195.0,195.0,488.0,31,36,30,12,12,1,6,7,12508.751326,1619.864045,10.909091,167.274024,471.994054,174.613165,0.000000,12.577415,6127.601421,866.789289,7.655240,68.916213,547.059733,11.787599,0.00000,40.400688,11806.0,44835.0,13866.0,70507.0,11360.0,44584.0,16384.0,72328.0,2016.25,5025.2,5265.4,6,"[0, 0, 2, 2, 2, 0]"
1,EL522,2011,23.139532,40.695359,0.265538,0.022865,0.253202,-0.253202,7.444165,11.375891,3.512438,-4.746063,5.242598,2.104167,193.713711,278.692913,836.078740,16.125000,20.083333,46.705426,865.533204,39.0,-4.0,43.0,10.333333,26.166667,26.500000,5.333333,1341.0,362.0,8.0,88.115950,512.0,157.0,210.0,391.0,31,30,30,10,10,1,6,6,8521.365773,2204.562938,6.102362,170.376578,294.050327,182.310346,0.006741,5.908999,5609.020284,824.307566,4.668086,44.086739,209.575010,6.368472,0.07597,10.388872,87330.0,375115.0,82051.0,544496.0,84172.0,402246.0,107376.0,593794.0,19627.51,9804.6,10058.8,10,"[0, 0, 3, 4, 3, 0]"
2,EL523,2011,22.758634,41.016242,0.193572,0.101583,0.277345,-0.277345,6.740102,10.879505,2.600698,-5.194396,5.133956,2.543045,233.660050,279.853480,839.560440,16.083333,21.500000,46.739130,890.054476,41.0,-5.0,46.0,9.833333,26.166667,27.000000,5.000000,1417.0,379.0,10.0,78.775725,584.0,198.0,223.0,377.0,31,36,30,12,12,3,5,8,10953.820049,1566.657477,6.461538,153.325412,268.365723,177.491290,0.028224,21.371767,4323.473197,1069.109617,4.638744,47.658551,207.195696,11.096824,0.15371,39.002021,5963.0,24603.0,9769.0,40335.0,5680.0,23436.0,11889.0,41005.0,1124.52,5809.2,5920.4,0,"[0, 0, 0, 0, 0, 0]"
3,EL524,2011,22.109199,40.888821,0.252725,0.133880,0.354723,-0.354723,6.226510,10.217141,2.235878,-5.870000,4.123182,3.739724,343.464625,308.363636,925.090909,14.750000,19.166667,47.916667,790.425662,36.0,-4.0,40.0,9.333333,24.500000,24.500000,4.666667,1630.0,348.0,19.0,63.316479,629.0,219.0,219.0,490.0,31,20,30,12,9,4,4,4,17893.740654,1234.209371,10.625000,142.505681,490.492098,175.009793,0.000000,16.377415,8324.265122,733.091339,9.669614,58.636781,412.361568,15.303624,0.00000,46.596750,11159.0,44537.0,14694.0,70390.0,11052.0,43212.0,17690.0,71954.0,1938.34,5025.2,5265.4,8,"[0, 0, 0, 3, 5, 0]"
4,EL525,2011,22.441443,40.269748,0.243297,0.118603,0.330566,-0.330566,7.617001,11.342695,3.891308,-4.498444,5.272222,2.673776,245.693081,484.000000,1452.000000,15.916667,18.333333,45.833333,769.247960,37.0,-3.0,40.0,15.666667,25.500000,25.500000,6.500000,1470.0,325.0,27.0,70.741701,556.0,174.0,174.0,437.0,31,36,30,12,12,1,6,7,11735.919071,971.980212,13.400000,134.587334,430.793162,170.687146,0.000000,21.108785,5383.969555,789.936868,8.881544,48.186142,455.707394,9.701413,0.00000,46.416540,10433.0,40170.0,12062.0,62665.0,10152.0,40273.0,14620.0,65045.0,1725.03,5025.2,5265.4,0,"[0, 0, 0, 0, 0, 0]"


In [5]:
cases_per_year = []

for year in dataset.year.unique():
    dataset_part = dataset[dataset.year == year]
    total_cases_part = dataset_part.cases_total.sum()
    row_part = {'year':year, 'cases':total_cases_part}
    cases_per_year.append(row_part)

cases_per_year_df = pd.DataFrame(cases_per_year)

cases_per_year_df

,year,cases
0,2011,91
1,2012,65
2,2013,56
3,2014,2
4,2015,0
5,2016,0
6,2017,0
7,2018,279
8,2019,130
9,2020,105


In [6]:
dataset['cases_arr'] = dataset['cases_arr'].apply(lambda x: json.loads(x))
dataset['cases_arr'] = dataset['cases_arr'].apply(lambda x: np.array(x))
dataset = convert_multiple_cases_array(dataset, target_col='cases_arr')
dataset['cases_total'] = dataset['cases_total'].apply(lambda x: 1 if x > 0 else 0)
print(dataset.shape)


(204, 73)


In [7]:
dataset.head()

,NUTS3_ID,year,x,y,ndvi_p1,ndwi_p1,ndmi_p1,ndbi_p1,lst_mean_p1,lst_day_mean_p1,lst_night_mean_p1,lst_min_p1,lst_max_p1,prec_mean_p1,prec_acc_p1,mosq_mean_p1,mosq_sum_p1,bio1,bio2,bio3,bio4,bio5,bio6,bio7,bio8,bio9,bio10,bio11,bio12,bio13,bio14,bio15,bio16,bio17,bio18,bio19,lc_prop1,lc_prop2,lc_prop3,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,distance_to_coast_std,distance_to_river_std,slope_mean_1km_std,aspect_mean_200m_std,elevation_mean_1km_std,hillshade_mean_1km_std,fs_area_1km_std,flow_accu_200m_std,males_lt15,males_15t64,males_gt65,males_total,females_lt15,females_15t64,females_gt65,females_total,mio_eur,l_total,unl_total,cases_total,cases_arr
0,EL521,2011,22.215876,40.552658,0.252698,0.164174,0.372403,-0.372403,6.208697,9.993104,2.424290,-5.821636,3.715091,3.286000,301.706854,433.757576,1301.272727,14.583333,18.666667,49.122807,768.952455,34.0,-4.0,38.0,9.166667,23.833333,23.833333,4.833333,1580.0,347.0,21.0,69.297060,585.0,195.0,195.0,488.0,31,36,30,12,12,1,6,7,12508.751326,1619.864045,10.909091,167.274024,471.994054,174.613165,0.000000,12.577415,6127.601421,866.789289,7.655240,68.916213,547.059733,11.787599,0.00000,40.400688,11806.0,44835.0,13866.0,70507.0,11360.0,44584.0,16384.0,72328.0,2016.25,5025.2,5265.4,1,"[0, 0, 1, 1, 1, 0]"
1,EL522,2011,23.139532,40.695359,0.265538,0.022865,0.253202,-0.253202,7.444165,11.375891,3.512438,-4.746063,5.242598,2.104167,193.713711,278.692913,836.078740,16.125000,20.083333,46.705426,865.533204,39.0,-4.0,43.0,10.333333,26.166667,26.500000,5.333333,1341.0,362.0,8.0,88.115950,512.0,157.0,210.0,391.0,31,30,30,10,10,1,6,6,8521.365773,2204.562938,6.102362,170.376578,294.050327,182.310346,0.006741,5.908999,5609.020284,824.307566,4.668086,44.086739,209.575010,6.368472,0.07597,10.388872,87330.0,375115.0,82051.0,544496.0,84172.0,402246.0,107376.0,593794.0,19627.51,9804.6,10058.8,1,"[0, 0, 1, 1, 1, 0]"
2,EL523,2011,22.758634,41.016242,0.193572,0.101583,0.277345,-0.277345,6.740102,10.879505,2.600698,-5.194396,5.133956,2.543045,233.660050,279.853480,839.560440,16.083333,21.500000,46.739130,890.054476,41.0,-5.0,46.0,9.833333,26.166667,27.000000,5.000000,1417.0,379.0,10.0,78.775725,584.0,198.0,223.0,377.0,31,36,30,12,12,3,5,8,10953.820049,1566.657477,6.461538,153.325412,268.365723,177.491290,0.028224,21.371767,4323.473197,1069.109617,4.638744,47.658551,207.195696,11.096824,0.15371,39.002021,5963.0,24603.0,9769.0,40335.0,5680.0,23436.0,11889.0,41005.0,1124.52,5809.2,5920.4,0,"[0, 0, 0, 0, 0, 0]"
3,EL524,2011,22.109199,40.888821,0.252725,0.133880,0.354723,-0.354723,6.226510,10.217141,2.235878,-5.870000,4.123182,3.739724,343.464625,308.363636,925.090909,14.750000,19.166667,47.916667,790.425662,36.0,-4.0,40.0,9.333333,24.500000,24.500000,4.666667,1630.0,348.0,19.0,63.316479,629.0,219.0,219.0,490.0,31,20,30,12,9,4,4,4,17893.740654,1234.209371,10.625000,142.505681,490.492098,175.009793,0.000000,16.377415,8324.265122,733.091339,9.669614,58.636781,412.361568,15.303624,0.00000,46.596750,11159.0,44537.0,14694.0,70390.0,11052.0,43212.0,17690.0,71954.0,1938.34,5025.2,5265.4,1,"[0, 0, 0, 1, 1, 0]"
4,EL525,2011,22.441443,40.269748,0.243297,0.118603,0.330566,-0.330566,7.617001,11.342695,3.891308,-4.498444,5.272222,2.673776,245.693081,484.000000,1452.000000,15.916667,18.333333,45.833333,769.247960,37.0,-3.0,40.0,15.666667,25.500000,25.500000,6.500000,1470.0,325.0,27.0,70.741701,556.0,174.0,174.0,437.0,31,36,30,12,12,1,6,7,11735.919071,971.980212,13.400000,134.587334,430.793162,170.687146,0.000000,21.108785,5383.969555,789.936868,8.881544,48.186142,455.707394,9.701413,0.00000,46.416540,10433.0,40170.0,12062.0,62665.0,10152.0,40273.0,14620.0,65045.0,1725.03,5025.2,5265.4,0,"[0, 0, 0, 0, 0, 0]"


In [8]:
X = dataset.drop(columns=['cases_total','cases_arr'])
y = dataset['cases_arr']
s = dataset['cases_total']

In [9]:
#train_idx, test_idx, _, _ = train_test_split(X.index, y, stratify = s, test_size = 0.3, random_state = 0)
#train_idx, test_idx, _, _ = train_test_split(X.index, y, test_size = 0.3, random_state = 0)

year_out = [2021, 2022]

train_idx = X[~X['year'].isin(year_out)].index
test_idx = X[X['year'].isin(year_out)].index

In [10]:
print(f"Training set 0/1 Ratio: {(dataset.iloc[train_idx]['cases_total'].value_counts().get(0) / dataset.iloc[train_idx]['cases_total'].value_counts().get(1)):.2f}")
print(f"Test set 0/1 Ratio: {(dataset.iloc[test_idx]['cases_total'].value_counts().get(0) / dataset.iloc[test_idx]['cases_total'].value_counts().get(1)):.2f}")

Training set 0/1 Ratio: 1.43
Test set 0/1 Ratio: 1.43


In [11]:
features_to_remove = ['x', 'y', 'year']
feature_names = X.select_dtypes(exclude=['object']).drop(columns = features_to_remove).columns
print(feature_names)

Index(['ndvi_p1', 'ndwi_p1', 'ndmi_p1', 'ndbi_p1', 'lst_mean_p1',
       'lst_day_mean_p1', 'lst_night_mean_p1', 'lst_min_p1', 'lst_max_p1',
       'prec_mean_p1', 'prec_acc_p1', 'mosq_mean_p1', 'mosq_sum_p1', 'bio1',
       'bio2', 'bio3', 'bio4', 'bio5', 'bio6', 'bio7', 'bio8', 'bio9', 'bio10',
       'bio11', 'bio12', 'bio13', 'bio14', 'bio15', 'bio16', 'bio17', 'bio18',
       'bio19', 'lc_prop1', 'lc_prop2', 'lc_prop3', 'lc_type1', 'lc_type2',
       'lc_type3', 'lc_type4', 'lc_type5', 'distance_to_coast',
       'distance_to_river', 'slope_mean_1km', 'aspect_mean_200m',
       'elevation_mean_1km', 'hillshade_mean_1km', 'fs_area_1km',
       'flow_accu_200m', 'distance_to_coast_std', 'distance_to_river_std',
       'slope_mean_1km_std', 'aspect_mean_200m_std', 'elevation_mean_1km_std',
       'hillshade_mean_1km_std', 'fs_area_1km_std', 'flow_accu_200m_std',
       'males_lt15', 'males_15t64', 'males_gt65', 'males_total',
       'females_lt15', 'females_15t64', 'females_gt65', 'f

In [12]:
dataset_train = dataset.iloc[train_idx]
dataset_test = dataset.iloc[test_idx]

print(dataset_train.year.value_counts())
print(dataset_test.year.value_counts())

2011    17
2012    17
2013    17
2014    17
2015    17
2016    17
2017    17
2018    17
2019    17
2020    17
Name: year, dtype: int64
2021    17
2022    17
Name: year, dtype: int64


In [13]:
X_train = X.iloc[train_idx].select_dtypes(exclude=['object']).drop(columns = features_to_remove)
X_test = X.iloc[test_idx].select_dtypes(exclude=['object']).drop(columns = features_to_remove)

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [14]:
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
        
imputer = KNNImputer()
imputer.fit(X_train)
X_train_imputed = imputer.transform(X_train_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

y_train_np = np.stack(y_train.to_numpy())
y_test_np = np.stack(y_test.to_numpy())

In [15]:
## XGBoost Model Parameters

params = {
    'max_depth': 3,
    'verbosity': 0,
    'seed': 0,
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
}

In [16]:
model = xgb.XGBClassifier(**params)   
model_multiout =  ClassifierChain(model)
trained_model = model_multiout.fit(X_train_imputed, y_train_np)

# explainer = shap.KernelExplainer(trained_model.predict_proba, X_train_imputed, link = 'logit')
# explainer = shap.KernelExplainer(trained_model.predict_proba, shap.kmeans(X_train_imputed, 20), link = 'logit')
# shap_values = explainer.shap_values(X_train_imputed)

In [17]:
y_train_prob = trained_model.predict_proba(X_train_imputed)
y_test_prob = trained_model.predict_proba(X_test_imputed)

In [18]:
train_loss = log_loss_vector(y_train_np, y_train_prob)
print(f"Train Logloss: {train_loss:.4f}")

test_loss = log_loss_vector(y_test_np, y_test_prob)
print(f"Test Loss: {test_loss:.4f}")

Train Logloss: 0.0203
Test Loss: 0.5290


In [19]:
classification_thresholds = optimal_threashold_fbeta_vector(y_train_prob, y_train_np, round_factor = 3)
print(classification_thresholds)

[0.044 0.567 0.875 0.874 0.856 0.829]


In [20]:
classification_thresholds_test = optimal_threashold_fbeta_vector(y_test_prob, y_test_np, round_factor = 3)
print(classification_thresholds_test)

[0.017 0.235 0.944 0.811 0.059 0.008]


In [21]:
y_train_pred = np.where(y_train_prob < classification_thresholds, 0, 1)
y_test_pred = np.where(y_test_prob < classification_thresholds, 0, 1)

In [22]:
y_train_pred_with_test = np.where(y_train_prob < classification_thresholds_test, 0, 1)
y_test_pred_with_test = np.where(y_test_prob < classification_thresholds_test, 0, 1)

In [23]:
hm_loss = hamming_loss(y_test_np, y_test_pred)
print(f"Hamming Loss: {hm_loss}")

Hamming Loss: 0.12745098039215685


In [24]:
print(classification_report_multiout(y_test_np, y_test_pred, output_dict=False, target_names=['Month 5', 'Month 6', 'Month 7', 'Month 8', 'Month 9', 'Month 10']))

              precision    recall  f1-score   support

     Month 5       0.00      0.00      0.00         0
     Month 6       0.00      0.00      0.00         0
     Month 7       0.67      0.67      0.67         6
     Month 8       0.91      0.71      0.80        14
     Month 9       0.64      0.58      0.61        12
    Month 10       0.00      0.00      0.00         8

   micro avg       0.75      0.53      0.62        40
   macro avg       0.37      0.33      0.35        40
weighted avg       0.61      0.53      0.56        40
 samples avg       0.26      0.21      0.23        40



In [25]:
print(classification_report_multiout(y_test_np, y_test_pred_with_test, output_dict=False, target_names=['Month 5', 'Month 6', 'Month 7', 'Month 8', 'Month 9', 'Month 10']))

              precision    recall  f1-score   support

     Month 5       0.00      0.00      0.00         0
     Month 6       0.00      0.00      0.00         0
     Month 7       1.00      0.67      0.80         6
     Month 8       0.79      0.79      0.79        14
     Month 9       0.55      1.00      0.71        12
    Month 10       0.28      0.62      0.38         8

   micro avg       0.55      0.80      0.65        40
   macro avg       0.43      0.51      0.45        40
weighted avg       0.64      0.80      0.68        40
 samples avg       0.35      0.33      0.32        40



In [26]:
'''
MCM = [[TN, FP],
       [FN, TP]]
'''
labels = [0,1,2,3,4,5]

cm = multilabel_confusion_matrix(y_test_np, y_test_pred, labels=labels)

print(cm)

[[[34  0]
  [ 0  0]]

 [[34  0]
  [ 0  0]]

 [[26  2]
  [ 2  4]]

 [[19  1]
  [ 4 10]]

 [[18  4]
  [ 5  7]]

 [[26  0]
  [ 8  0]]]


In [27]:
cm = multilabel_confusion_matrix(y_test_np, y_test_pred_with_test, labels=labels)

print(cm)


[[[34  0]
  [ 0  0]]

 [[34  0]
  [ 0  0]]

 [[28  0]
  [ 2  4]]

 [[17  3]
  [ 3 11]]

 [[12 10]
  [ 0 12]]

 [[13 13]
  [ 3  5]]]


In [28]:
# shap.summary_plot(shap_values = shap_values, features = feature_names)